# Effect sizes, confidence intervals and multiple-comparison corrections
### GAT, GATv2 and GAT+GSL vs Random Forest, including AUROC and AUPRC

Addresses the reviewer request to distinguish **statistical** from **practical** significance.

**Inputs** (same folder as this notebook; the files exported by the t-test notebook, named by variant):

| Variant | files |
|---|---|
| GAT | `FINAL_per_class_ttest_vs_RF.csv`, `FINAL_overall_ttest_vs_RF.csv`, `FINAL_auc_ttest_vs_RF.csv` |
| GATv2 | `FINAL_gatv2_per_class_ttest_vs_RF.csv`, `FINAL_gatv2_overall_ttest_vs_RF.csv`, `FINAL_gatv2_auc_ttest_vs_RF.csv` |
| GAT+GSL | `FINAL_gatGSL_per_class_ttest_vs_RF.csv`, `FINAL_gatGSL_overall_ttest_vs_RF.csv`, `FINAL_gatGSL_auc_ttest_vs_RF.csv` |

**What it computes**, for every architecture (GCN, GAT, GINE, GraphSAGE) in each variant set, against the fixed Random Forest reference:
- absolute difference (GNN - RF) with a **95% CI** (t-based, df = 9) and **Hedges' g**;
- raw p, **Holm** (primary), Benjamini-Hochberg q (sensitivity) and the previous Bonferroni-within-family p;
- a **practical-significance verdict** against a minimal relevant difference (MID) that you must choose and justify;
- a **robustness check**: RF's classification metrics are stored to 2 decimals, so a result is flagged if its CI comes within 0.005 of zero.

**Two families of tests per variant.** *Primary*: the 56 classification-metric comparisons (accuracy, MCC, precision/recall/F1 per class).
*Secondary*: the 40 AUROC/AUPRC comparisons (macro + 4 classes, 2 metrics, 4 architectures). Holm and BH are applied within each family, so adding
AUROC/AUPRC does not change the primary results; `p_holm_all` (one family of all 96) is reported as a conservative sensitivity check.

GATv2 and GAT+GSL are also compared with plain GAT (Welch test from the mean/SD summaries, 10 vs 10 runs) on 8 primary endpoints, with Holm over the 16 comparisons.

**Caveats to state in the manuscript**
1. CIs and p-values reflect **training-seed variability only**, not uncertainty from the finite test set (226 molecules; only 11 IA). AUPRC for IA is especially noisy.
2. Hedges' g is in units of seed-to-seed SD (small), so values are inflated; lead with the absolute difference and its CI.
3. The MID is a domain judgement, not a statistical quantity. AUROC and AUPRC are on different scales (AUPRC chance level = class prevalence), so one MID may not suit both.
4. AUROC/AUPRC are threshold-free: they compare how well models *rank* molecules, whereas F1 and recall also reflect each model's decision threshold.

## 1. Statistical functions and plotting helpers

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from scipy import stats

N_RUNS = 10
CLASSES = ["AA", "AI", "IA", "II"]
N_TEST = 226                                                # test-set size
SUPPORTS = {"AA": 58, "AI": 82, "IA": 11, "II": 75}         # test-set class counts (scaffold split)
FAMILY_CLS = "classification metrics (primary)"             # Accuracy, MCC, precision/recall/F1 per class  -> 56 tests / variant
FAMILY_AUC = "AUROC/AUPRC (secondary)"                      # macro + per-class AUROC and AUPRC             -> 40 tests / variant
METRIC_LABEL = {"precision": "Precision", "recall": "Recall", "f1": "F1", "AUROC": "AUROC", "AUPRC": "AUPRC"}


def one_sample_stats(mu0, xbar, s_pop, n=N_RUNS, conf=0.95):
    """GNN n-run distribution (mean xbar, population SD s_pop) vs a fixed RF value mu0."""
    if s_pop == 0 or np.isnan(s_pop):
        return dict(diff=xbar - mu0, ci_low=np.nan, ci_high=np.nan, t=np.nan, p=np.nan, hedges_g=np.nan)
    df = n - 1
    s = s_pop * np.sqrt(n / (n - 1))
    se = s / np.sqrt(n)
    diff = xbar - mu0
    t = diff / se
    p = 2 * (1 - stats.t.cdf(abs(t), df))
    tcrit = stats.t.ppf(1 - (1 - conf) / 2, df)
    g = (diff / s) * (1 - 3 / (4 * df - 1))
    return dict(diff=diff, ci_low=diff - tcrit * se, ci_high=diff + tcrit * se, t=t, p=p, hedges_g=g)


def welch_from_summary(m1, sd1_pop, m2, sd2_pop, n=N_RUNS, conf=0.95):
    """Two-sample Welch comparison (variant 1 minus variant 2) from mean / population-SD summaries."""
    s1 = sd1_pop * np.sqrt(n / (n - 1)); s2 = sd2_pop * np.sqrt(n / (n - 1))
    v1, v2 = s1 ** 2 / n, s2 ** 2 / n
    diff = m1 - m2
    se = np.sqrt(v1 + v2)
    df = (v1 + v2) ** 2 / (v1 ** 2 / (n - 1) + v2 ** 2 / (n - 1))
    t = diff / se
    p = 2 * (1 - stats.t.cdf(abs(t), df))
    tcrit = stats.t.ppf(1 - (1 - conf) / 2, df)
    sp = np.sqrt((s1 ** 2 + s2 ** 2) / 2)
    g = (diff / sp) * (1 - 3 / (4 * (2 * n - 2) - 1))
    return dict(diff=diff, ci_low=diff - tcrit * se, ci_high=diff + tcrit * se, t=t, df=df, p=p, hedges_g=g)


def holm(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    pv = p[ok]; m = len(pv); order = np.argsort(pv); adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * pv[idx]); adj[idx] = min(1.0, run)
    out[ok] = adj
    return out


def benjamini_hochberg(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    pv = p[ok]; m = len(pv); order = np.argsort(pv)[::-1]; adj = np.empty(m); run = 1.0
    for rank, idx in enumerate(order):
        run = min(run, pv[idx] * m / (m - rank)); adj[idx] = run
    out[ok] = adj
    return out


def _decimals(x):
    s = f"{float(x):.10f}".rstrip("0")
    return len(s.split(".")[1]) if "." in s else 0


def load_variant(per_class_csv, overall_csv, auc_csv):
    """One long table: Architecture, Family, Scope, Class, Metric, Endpoint, RF, GNN_mean, GNN_std."""
    cols = ["Architecture", "Family", "Scope", "Class", "Metric", "RF", "GNN_mean", "GNN_std"]
    pc = pd.read_csv(per_class_csv); ov = pd.read_csv(overall_csv); au = pd.read_csv(auc_csv)
    a = ov.assign(Family=FAMILY_CLS, Scope="overall", Class="-")[cols]
    b = pc.assign(Family=FAMILY_CLS, Scope="per-class")[cols]
    c = au.assign(Family=FAMILY_AUC, Scope=np.where(au.Class == "macro", "overall", "per-class"))[cols]
    df = pd.concat([a, b, c], ignore_index=True)

    def endpoint(r):
        if r.Scope == "overall":
            return r.Metric if r.Family == FAMILY_CLS else f"Macro {r.Metric}"
        return f"{METRIC_LABEL[r.Metric]} - {r.Class}"
    df["Endpoint"] = df.apply(endpoint, axis=1)
    return df


def vs_rf(long_df, mid=0.03, mid_by_metric=None):
    """Every architecture vs the fixed RF reference, for every endpoint of one GAT variant."""
    mid_by_metric = mid_by_metric or {}
    rows = [{**r.to_dict(), **one_sample_stats(r.RF, r.GNN_mean, r.GNN_std)} for _, r in long_df.iterrows()]
    df = pd.DataFrame(rows)

    # previous rule, kept for continuity: Bonferroni within (architecture x metric) across the 4 classes; x2 for the 2 macro AUC endpoints
    df["p_bonf_within_family"] = np.nan
    pcm = df.Scope == "per-class"
    for _, idx in df[pcm].groupby(["Architecture", "Metric"]).groups.items():
        df.loc[idx, "p_bonf_within_family"] = np.minimum(1.0, df.loc[idx, "p"] * 4)
    macro_auc = (df.Family == FAMILY_AUC) & (df.Scope == "overall")
    for _, idx in df[macro_auc].groupby("Architecture").groups.items():
        df.loc[idx, "p_bonf_within_family"] = np.minimum(1.0, df.loc[idx, "p"] * 2)

    # PRIMARY: Holm within each family (56 classification tests; 40 AUROC/AUPRC tests); BH as sensitivity
    df["p_holm"] = np.nan; df["q_bh"] = np.nan
    for fam, idx in df.groupby("Family").groups.items():
        df.loc[idx, "p_holm"] = holm(df.loc[idx, "p"].values)
        df.loc[idx, "q_bh"] = benjamini_hochberg(df.loc[idx, "p"].values)
    df["p_holm_all"] = holm(df.p.values)                       # conservative sensitivity: one family of all 96 tests
    df["sig_holm"] = df.p_holm < 0.05
    df["sig_holm_all"] = df.p_holm_all < 0.05

    # robustness to the precision of the RF reference (RF classification metrics are rounded to 2 decimals)
    pad = {}
    for fam, g in df.groupby("Family"):
        dec = max(_decimals(x) for x in g.RF.unique())
        pad[fam] = 0.5 * 10 ** (-dec) if dec <= 4 else 0.0
    df["rf_rounding_pad"] = df.Family.map(pad)
    df["robust_to_RF_rounding"] = (df.ci_low > df.rf_rounding_pad) | (df.ci_high < -df.rf_rounding_pad)

    df["acc_diff_in_molecules"] = np.where(df.Metric == "Accuracy", df["diff"] * N_TEST, np.nan)
    df["recall_diff_in_molecules"] = np.where(df.Metric == "recall", df["diff"] * df.Class.map(SUPPORTS), np.nan)

    def verdict(r):
        if not r.sig_holm:
            return "not significant"
        m = mid_by_metric.get(r.Metric, mid)
        return "significant, practically meaningful" if abs(r["diff"]) >= m else "significant but small"
    df["verdict"] = df.apply(verdict, axis=1)
    return df


PRIMARY_ENDPOINTS = ["Accuracy", "MCC", "Macro AUROC", "Macro AUPRC"] + [f"F1 - {c}" for c in CLASSES]


def variants_vs_base(long_by_variant, base="GAT", arch="GAT", endpoints=PRIMARY_ENDPOINTS):
    """GATv2 / GAT+GSL vs plain GAT: Welch from summary stats, on the primary endpoints; Holm over all comparisons."""
    def pick(ldf):
        d = ldf[(ldf.Architecture == arch)].set_index("Endpoint")
        return {e: d.loc[e] for e in endpoints}
    base_ep = pick(long_by_variant[base]); rows = []
    for v, ldf in long_by_variant.items():
        if v == base:
            continue
        ep = pick(ldf)
        for name in endpoints:
            r, b = ep[name], base_ep[name]
            w = welch_from_summary(r.GNN_mean, r.GNN_std, b.GNN_mean, b.GNN_std)
            rows.append(dict(Variant=v, Endpoint=name, variant_mean=r.GNN_mean, GAT_mean=b.GNN_mean, **w))
    df = pd.DataFrame(rows)
    df["p_holm"] = holm(df.p.values); df["q_bh"] = benjamini_hochberg(df.p.values)
    df["sig_holm"] = df.p_holm < 0.05
    return df


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARCH_COLORS = {"GCN": "#94a3b8", "GAT": "#2563eb", "GINE": "#f59e0b", "GraphSAGE": "#ef4444"}
CLASSES = ["AA", "AI", "IA", "II"]


def forest_vs_rf(df, mid=0.03, title="", save_path="Figure_effect_size_forest_vs_RF.png"):
    """Four panels: overall endpoints, then per-class F1, AUROC and AUPRC. Filled marker = significant (Holm within its family)."""
    archs = list(ARCH_COLORS)
    panels = [("A. Overall: accuracy, MCC, macro AUROC, macro AUPRC", ["Accuracy", "MCC", "Macro AUROC", "Macro AUPRC"]),
              ("B. Per-class F1-score", [f"F1 - {c}" for c in CLASSES]),
              ("C. Per-class AUROC (threshold-free)", [f"AUROC - {c}" for c in CLASSES]),
              ("D. Per-class AUPRC (threshold-free)", [f"AUPRC - {c}" for c in CLASSES])]
    fig, axes = plt.subplots(2, 2, figsize=(15, 13))
    for ax, (ttl, endpoints) in zip(axes.ravel(), panels):
        y = 0; ticks = []
        ax.axvspan(-mid, mid, color="#e2e8f0", alpha=0.7, zorder=0); ax.axvline(0, color="black", lw=1)
        for ep in endpoints:
            start = y
            for a in archs:
                r = df[(df.Endpoint == ep) & (df.Architecture == a)].iloc[0]
                c = ARCH_COLORS[a]; sig = bool(r.sig_holm)
                ax.errorbar(r["diff"], y, xerr=[[r["diff"] - r.ci_low], [r.ci_high - r["diff"]]], fmt="o", color=c, ecolor=c,
                            elinewidth=2, capsize=3, markersize=7, markerfacecolor=c if sig else "white",
                            markeredgewidth=1.8, zorder=3)
                y += 1
            ticks.append((start + (len(archs) - 1) / 2, ep.split(" - ")[-1] if " - " in ep else ep)); y += 0.6
        ax.set_yticks([p for p, _ in ticks]); ax.set_yticklabels([l for _, l in ticks], fontsize=11); ax.invert_yaxis()
        ax.set_xlabel("Difference from Random Forest (GNN − RF), 95% CI")
        ax.set_title(ttl, fontweight="bold", loc="left", fontsize=12)
        ax.grid(axis="x", alpha=0.3); ax.spines[["top", "right"]].set_visible(False)

    handles = [plt.Line2D([0], [0], marker="o", color=ARCH_COLORS[a], lw=0, markersize=8, label=a) for a in archs]
    handles += [plt.Line2D([0], [0], marker="o", color="k", markerfacecolor="k", lw=0, markersize=8, label="significant (Holm, within family)"),
                plt.Line2D([0], [0], marker="o", color="k", markerfacecolor="white", lw=0, markersize=8, label="not significant"),
                plt.Rectangle((0, 0), 1, 1, color="#e2e8f0", label=f"±{mid:.2f} band")]
    fig.legend(handles=handles, loc="lower center", ncol=7, frameon=False, fontsize=10)
    fig.suptitle(title or "Effect sizes with 95% CIs vs. Random Forest (10 seeded runs per architecture)", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    fig.savefig(save_path, dpi=200, bbox_inches="tight"); plt.close(fig)


def forest_variants(vb, mid=0.03, save_path="Figure_variants_vs_plain_GAT.png"):
    colors = {"GATv2": "#0ea5e9", "GAT_GSL": "#8b5cf6"}
    endpoints = list(dict.fromkeys(vb.Endpoint))
    fig, ax = plt.subplots(figsize=(8.5, 7))
    ax.axvspan(-mid, mid, color="#e2e8f0", alpha=0.7, zorder=0); ax.axvline(0, color="black", lw=1)
    y = 0; ticks = []
    for ep in endpoints:
        start = y
        for v in colors:
            r = vb[(vb.Endpoint == ep) & (vb.Variant == v)].iloc[0]; sig = bool(r.sig_holm)
            ax.errorbar(r["diff"], y, xerr=[[r["diff"] - r.ci_low], [r.ci_high - r["diff"]]], fmt="o", color=colors[v], ecolor=colors[v],
                        elinewidth=2, capsize=3, markersize=7, markerfacecolor=colors[v] if sig else "white", markeredgewidth=1.8, zorder=3)
            y += 1
        ticks.append((start + 0.5, ep)); y += 0.6
    ax.set_yticks([p for p, _ in ticks]); ax.set_yticklabels([l for _, l in ticks]); ax.invert_yaxis()
    ax.set_xlabel("Difference from plain GAT (variant − GAT), 95% CI")
    ax.set_title("GATv2 and GAT+GSL relative to plain GAT (10 seeded runs each)", fontweight="bold", loc="left")
    ax.grid(axis="x", alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
    handles = [plt.Line2D([0], [0], marker="o", color=c, lw=0, markersize=8, label=v) for v, c in colors.items()]
    handles += [plt.Line2D([0], [0], marker="o", color="k", markerfacecolor="k", lw=0, markersize=8, label="significant (Holm)"),
                plt.Line2D([0], [0], marker="o", color="k", markerfacecolor="white", lw=0, markersize=8, label="not significant")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=4, fontsize=9, frameon=False)
    plt.tight_layout(); fig.savefig(save_path, dpi=200, bbox_inches="tight"); plt.close(fig)


## 2. Configuration

In [ ]:
INPUT_DIR = "."
VARIANT_PREFIX = {"GAT": "FINAL_", "GATv2": "FINAL_gatv2_", "GAT_GSL": "FINAL_gatGSL_"}   # label -> file-name prefix
MID = 0.03            # minimal practically relevant difference -- ADAPT and justify
MID_BY_METRIC = {}    # optional per-metric override, e.g. {"AUPRC": 0.05}
MAIN_VARIANT = "GAT"  # variant used for the manuscript table and the main forest plot

# N_TEST and SUPPORTS (test-set size and class counts) are defined in the functions cell above; update them there if the split changes.

## 3. Every architecture vs Random Forest, for each variant

In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.width", 260); pd.set_option("display.max_columns", 40)

long_by = {v: load_variant(f"{INPUT_DIR}/{p}per_class_ttest_vs_RF.csv", f"{INPUT_DIR}/{p}overall_ttest_vs_RF.csv",
                           f"{INPUT_DIR}/{p}auc_ttest_vs_RF.csv") for v, p in VARIANT_PREFIX.items()}
res = {v: vs_rf(long_by[v], mid=MID, mid_by_metric=MID_BY_METRIC) for v in VARIANT_PREFIX}
all_res = pd.concat([r.assign(Variant=v) for v, r in res.items()], ignore_index=True)
all_res.to_csv("effect_size_results_vs_RF_all_variants.csv", index=False)

d = res[MAIN_VARIANT]
cols = ["Architecture","Endpoint","RF","GNN_mean","diff","ci_low","ci_high","hedges_g","p","p_holm","verdict"]
print(f"=== {MAIN_VARIANT}: overall (Accuracy, MCC, macro AUROC, macro AUPRC) ===")
print(d[d.Scope == "overall"][cols + ["acc_diff_in_molecules"]].round(4).to_string(index=False))

In [ ]:
print(f"=== {MAIN_VARIANT}: per-class F1 ===");  print(d[d.Metric == "f1"][cols].round(4).to_string(index=False))
print(f"\n=== {MAIN_VARIANT}: per-class AUROC ===");  print(d[(d.Metric == "AUROC") & (d.Scope == "per-class")][cols].round(4).to_string(index=False))
print(f"\n=== {MAIN_VARIANT}: per-class AUPRC ===");  print(d[(d.Metric == "AUPRC") & (d.Scope == "per-class")][cols].round(4).to_string(index=False))
print(f"\n=== {MAIN_VARIANT}: per-class recall, with the difference in test-set molecules ===")
print(d[d.Metric == "recall"][["Architecture","Endpoint","RF","GNN_mean","diff","ci_low","ci_high","p_holm","recall_diff_in_molecules","verdict"]].round(4).to_string(index=False))

## 4. How much do the corrections matter, and are results robust?

In [ ]:
for v, r in res.items():
    for fam, g in r.groupby("Family"):
        print(f"{v:8s} {fam:34s} n={len(g):2d} | raw p<.05: {int((g.p<.05).sum()):2d} | Holm (family): {int(g.sig_holm.sum()):2d} | "
              f"BH q<.05: {int((g.q_bh<.05).sum()):2d} | Holm (all 96): {int(g.sig_holm_all.sum()):2d} | "
              f"Holm-significant but NOT rounding-robust: {int((g.sig_holm & ~g.robust_to_RF_rounding).sum())}")

print(f"\nHolm-significant within its family but NOT under the all-96 correction ({MAIN_VARIANT}):")
x = d[d.sig_holm & ~d.sig_holm_all][["Architecture","Endpoint","diff","p_holm","p_holm_all"]]
print(x.round(4).to_string(index=False) if len(x) else "  none")

print(f"\nVerdict counts vs MID ({MAIN_VARIANT}, all 96 comparisons):")
for m in [0.02, 0.03, 0.05]:
    d2 = vs_rf(long_by[MAIN_VARIANT], mid=m)
    print(f"  MID={m:.2f}: practically meaningful={int((d2.verdict=='significant, practically meaningful').sum())}, "
          f"significant but small={int((d2.verdict=='significant but small').sum())}")

## 5. GATv2 and GAT+GSL against plain GAT

In [ ]:
vb = variants_vs_base(long_by, base="GAT", arch="GAT")
vb.to_csv("variants_vs_plain_GAT.csv", index=False)
print(vb[["Variant","Endpoint","variant_mean","GAT_mean","diff","ci_low","ci_high","hedges_g","p","p_holm","sig_holm"]].round(4).to_string(index=False))

## 6. Manuscript-ready table (main variant; overall, F1, AUROC and AUPRC)

In [ ]:
def fmt_p(p): return "<0.001" if p < 0.001 else f"{p:.3f}"
order = ["Accuracy", "MCC", "Macro AUROC", "Macro AUPRC"] + [f"{m} - {c}" for m in ["F1", "AUROC", "AUPRC"] for c in CLASSES]
key = d[d.Endpoint.isin(order)].copy()
key["Endpoint"] = pd.Categorical(key.Endpoint, categories=order, ordered=True)
key["Architecture"] = pd.Categorical(key.Architecture, categories=["GCN", "GAT", "GINE", "GraphSAGE"], ordered=True)
key = key.sort_values(["Endpoint", "Architecture"])
table = pd.DataFrame({
    "Endpoint": key.Endpoint.astype(str), "Architecture": key.Architecture.astype(str), "Family": key.Family,
    "RF": key.RF.round(3),
    "GNN (mean +/- SD, 10 runs)": key.GNN_mean.round(3).astype(str) + " +/- " + key.GNN_std.round(3).astype(str),
    "Difference [95% CI]": key["diff"].map("{:+.3f}".format) + " [" + key.ci_low.map("{:+.3f}".format) + ", " + key.ci_high.map("{:+.3f}".format) + "]",
    "Hedges g": key.hedges_g.round(2), "p (raw)": key.p.map(fmt_p), "p (Holm, family)": key.p_holm.map(fmt_p),
    "q (BH, family)": key.q_bh.map(fmt_p), "p (Holm, all 96)": key.p_holm_all.map(fmt_p), "Interpretation": key.verdict,
}).reset_index(drop=True)
table.to_csv(f"effect_size_manuscript_table_{MAIN_VARIANT}.csv", index=False)
table

## 7. Figures

In [ ]:
forest_vs_rf(res[MAIN_VARIANT], mid=MID,
             title=f"{MAIN_VARIANT} set: effect sizes with 95% CIs vs. Random Forest (10 seeded runs per architecture)",
             save_path="Figure_effect_size_forest_vs_RF.png")
forest_variants(vb, mid=MID, save_path="Figure_variants_vs_plain_GAT.png")
from IPython.display import Image, display
display(Image("Figure_effect_size_forest_vs_RF.png")); display(Image("Figure_variants_vs_plain_GAT.png"))